In [20]:
import pandas as pd
import numpy as np
import warnings
from sklearn.ensemble import GradientBoostingClassifier
from collections import defaultdict

# 1. Suppress warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. ADVANCED ELO ENGINE (Unchanged)
# ==========================================
def calculate_elo(df):
    current_elo = defaultdict(lambda: 1500) 
    elo_history = []
    K_FACTOR = 30
    
    for idx, row in df.iterrows():
        home, away = row['Home'], row['Away']
        h_score, a_score = row['Home_Goals'], row['Away_Goals']
        
        h_elo = current_elo[home]
        a_elo = current_elo[away]
        elo_history.append({'Home_Elo': h_elo, 'Away_Elo': a_elo})
        
        expected_home = 1 / (1 + 10 ** ((a_elo - h_elo) / 400))
        
        if h_score > a_score: result = 1.0
        elif h_score == a_score: result = 0.5
        else: result = 0.0
            
        current_elo[home] = h_elo + K_FACTOR * (result - expected_home)
        current_elo[away] = a_elo + K_FACTOR * ((1 - result) - (1 - expected_home))
        
    return pd.concat([df, pd.DataFrame(elo_history)], axis=1), current_elo

# ==========================================
# 2. TRAIN MODEL
# ==========================================
try:
    df = pd.read_csv("AFCON_Matches_1990_Present.csv")
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date', ascending=True).reset_index(drop=True)
    df = df.rename(columns={'home_team': 'Home', 'away_team': 'Away', 'home_score': 'Home_Goals', 'away_score': 'Away_Goals'})
    
    df, final_ratings = calculate_elo(df)
    
    conditions = [
        (df['Home_Goals'] > df['Away_Goals']),
        (df['Home_Goals'] == df['Away_Goals']),
        (df['Home_Goals'] < df['Away_Goals'])
    ]
    df['Result'] = np.select(conditions, [2, 1, 0])
    df['Elo_Diff'] = df['Home_Elo'] - df['Away_Elo']
    
    model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
    model.fit(df[['Elo_Diff']], df['Result'])
    
except Exception as e:
    print(f"❌ Error: {e}")
    exit()

# ==========================================
# 3. PREDICT WITH CONSISTENCY CHECK
# ==========================================
def predict_match(team1, team2):
    # --- A. PREPARE DATA ---
    elo1 = final_ratings.get(team1, 1500)
    elo2 = final_ratings.get(team2, 1500)
    
    # Tournament Context Boosts
    form_boost = {
        'Morocco': 120,   # Massive Home Advantage
        'Senegal': 80,
        'Nigeria': 70,
        'Ivory Coast': 40,
        'Mali': 30,
        'Algeria': 30,
        'Cameroon': 10,
        'Egypt': -10
    }
    
    elo1 += form_boost.get(team1, 0)
    elo2 += form_boost.get(team2, 0)
    
    diff = elo1 - elo2
    X_pred = pd.DataFrame({'Elo_Diff': [diff]})
    
    # --- B. GET PROBABILITIES ---
    probs = model.predict_proba(X_pred)[0] # [Away, Draw, Home]
    
    # Find the most likely outcome
    # 0 = Team2 Win, 1 = Draw, 2 = Team1 Win
    predicted_outcome = np.argmax(probs) 
    
    # --- C. GENERATE CONSISTENT SCORE ---
    # We loop until the score matches the predicted outcome
    # This prevents "High Win Probability" but "Losing Score"
    
    xg_1 = max(0.4, 1.2 + (diff/450)) # Expected Goals Team 1
    xg_2 = max(0.4, 1.2 - (diff/450)) # Expected Goals Team 2
    
    while True:
        score_1 = np.random.poisson(xg_1)
        score_2 = np.random.poisson(xg_2)
        
        if predicted_outcome == 2:   # Team 1 should win
            if score_1 > score_2: break
        elif predicted_outcome == 0: # Team 2 should win
            if score_2 > score_1: break
        else:                        # Draw
            if score_1 == score_2: break
            
        # Safety break for rare loops (force a realistic result manually if loop gets stuck)
        if np.random.rand() < 0.05: 
            if predicted_outcome == 2: score_1, score_2 = int(xg_1)+1, int(xg_2)
            elif predicted_outcome == 0: score_1, score_2 = int(xg_1), int(xg_2)+1
            else: score_1 = score_2 = 1
            break

    print(f"\n⚽ {team1} vs {team2}")
    print(f"   Strength: {team1} ({int(elo1)}) vs {team2} ({int(elo2)})")
    print(f"   Win Probabilities: {team1} {probs[2]*100:.1f}% | Draw {probs[1]*100:.1f}% | {team2} {probs[0]*100:.1f}%")
    print(f"   Predicted Score: {team1} {score_1} – {score_2} {team2}")

print("\n🏆 --- CONSISTENT PREDICTIONS ---")
predict_match("Mali", "Senegal")
predict_match("Cameroon", "Morocco") 
predict_match("Nigeria", "Algeria")
predict_match("Ivory Coast", "Egypt")


🏆 --- CONSISTENT PREDICTIONS ---

⚽ Mali vs Senegal
   Strength: Mali (1598) vs Senegal (1703)
   Win Probabilities: Mali 17.5% | Draw 36.4% | Senegal 46.1%
   Predicted Score: Mali 1 – 3 Senegal

⚽ Cameroon vs Morocco
   Strength: Cameroon (1621) vs Morocco (1696)
   Win Probabilities: Cameroon 37.3% | Draw 19.8% | Morocco 42.9%
   Predicted Score: Cameroon 1 – 2 Morocco

⚽ Nigeria vs Algeria
   Strength: Nigeria (1746) vs Algeria (1535)
   Win Probabilities: Nigeria 91.8% | Draw 3.6% | Algeria 4.6%
   Predicted Score: Nigeria 3 – 1 Algeria

⚽ Ivory Coast vs Egypt
   Strength: Ivory Coast (1690) vs Egypt (1650)
   Win Probabilities: Ivory Coast 50.0% | Draw 27.9% | Egypt 22.2%
   Predicted Score: Ivory Coast 2 – 0 Egypt
